# Exploratory Data Analysis (EDA)

Before building features or training a churn model, we explore the CRM
snapshot to understand class balance, separability, and which subscription
attributes carry the churn signal — those become the feature view's schema.


## Build the prepared tables

`prepare_data.py` downloads the raw CSV when absent and writes the clean
interim parquet — idempotent, so it is safe to re-run.


In [ ]:
%%bash
cd .. && python -m telco_customer_churn.prepare_data

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_parquet("../data/interim/customers_clean.parquet")
df.shape, df.dtypes.head(6)

## Class Balance

Churn is an imbalanced problem — roughly one in four subscribers cancels —
so accuracy alone is a misleading metric; the model must rank, not just
threshold.


In [ ]:
churn_rate = df["churn"].mean()
counts = df["churn"].value_counts().sort_index()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(["stayed", "churned"], counts.values, color=["#458588", "#cc241d"])
ax[0].set_ylabel("customers")
ax[0].set_title("Class counts")
ax[1].hist(
    df.loc[df["churn"] == 0, "tenure"], bins=30, alpha=0.7, label="stayed"
)
ax[1].hist(
    df.loc[df["churn"] == 1, "tenure"], bins=30, alpha=0.7, label="churned"
)
ax[1].set_xlabel("tenure (months)")
ax[1].legend()
ax[1].set_title("Tenure distribution by outcome")
plt.tight_layout()
plt.show()
print(f"churn rate: {churn_rate:.1%}")

The tenure histogram is the dataset's signature shape: churn concentrates
heavily in the first contract months and decays with tenure — long-tenured
customers rarely leave. Early-tenure risk is exactly what the retention
team wants scored nightly.


## Contract Type — the Dominant Driver


In [ ]:
contract_churn = df.groupby("Contract")["churn"].agg(["mean", "count"])
ax = contract_churn["mean"].plot(kind="bar", color="#d79921", figsize=(7, 4))
ax.set_ylabel("churn rate")
ax.set_title("Churn rate by contract type")
plt.xticks(rotation=15)
plt.show()
contract_churn

Month-to-month customers churn at an order of magnitude higher rate than
one- and two-year commitments — contract type is why the feature view keeps
`month_to_month` and `one_year_contract` as separate indicators rather than
a single ordinal.


## Billing Signals


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df.boxplot(column="MonthlyCharges", by="churn", ax=ax[0])
ax[0].set_title("Monthly charges by outcome")
ax[0].set_xlabel("churned")
payment_churn = df.groupby("PaymentMethod")["churn"].mean().sort_values()
payment_churn.plot(kind="barh", ax=ax[1], color="#689d6a")
ax[1].set_xlabel("churn rate")
ax[1].set_title("Churn rate by payment method")
plt.tight_layout()
plt.show()

Higher monthly charges correlate with churn (price pressure), and
**electronic check** stands out among payment methods — a known proxy for
lower-commitment customers in this dataset. Both signals enter the feature
contract (`monthly_charges`, `electronic_check`).


## Service Add-ons


In [ ]:
add_on_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]
add_on_churn = (
    df[add_on_cols]
    .eq("Yes")
    .groupby(df["churn"])
    .mean()
    .T.rename(columns={0: "stayed", 1: "churned"})
)
ax = add_on_churn.plot(kind="bar", figsize=(9, 4))
ax.set_ylabel("subscription rate")
ax.set_title("Add-on subscription rates by outcome")
plt.xticks(rotation=30)
plt.show()

Stayed customers carry more protection/support add-ons (security, backup,
tech support) while streaming services barely separate the classes —
exactly why the feature contract consolidates add-ons into `num_add_ons`
while keeping `streaming_tv`/`streaming_movies` individually.

## Takeaways for Feature Engineering

- `tenure_months`, `month_to_month`, `one_year_contract` — primary signals.
- `monthly_charges`, `total_charges`, `electronic_check` — billing pressure.
- `num_add_ons` — consolidated engagement depth.
- Class imbalance (~26% churn) → AUC-focused evaluation at the model stage.
